# Week 4 — Real-schema SQLite Query Benchmarks

This is an end-to-end benchmark of the repository's actual Week 4 SQLAlchemy schema and loader. It loads `outputs/profile_cleaned_features.parquet` through `ingen_pydev.db.loader.load_week3_output`, preserves the primary keys and the `uq_sensor_reading_device_timestamp` unique constraint, measures five queries, adds only supplemental benchmark indexes, repeats the measurements, captures `EXPLAIN QUERY PLAN`, and verifies exact result equality.

The 10,000-row profile is scaled without inventing telemetry values: 1, 10, and 100 logical devices load the same real profile, producing 10,000, 100,000, and 1,000,000 `sensor_reading` rows. Device and session identities differ, while the measured telemetry remains the profile output. Loader time is reported because this is an end-to-end build, but query latency is measured separately.

In [26]:
from __future__ import annotations

import gc
import importlib
import json
import sqlite3
import statistics
import time
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

import ingen_pydev.db.loader as loader_module

PARQUET_PATH = Path("outputs/profile_cleaned_features.parquet")
SUMMARY_PATH = Path("outputs/profile_validation_summary.json")
OUTPUT_DIR = Path("outputs/w04_query_benchmarks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_ROWS = (10_000, 100_000, 1_000_000)

assert PARQUET_PATH.exists(), PARQUET_PATH
assert SUMMARY_PATH.exists(), SUMMARY_PATH
profile_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
PROFILE_ROWS = int(profile_summary["rows_processed"])
assert PROFILE_ROWS == 10_000
assert all(target % PROFILE_ROWS == 0 for target in TARGET_ROWS)
print(f"Source: {PARQUET_PATH.resolve()} ({PROFILE_ROWS:,} rows)")

Source: C:\ingen\outputs\profile_cleaned_features.parquet (10,000 rows)


## Real loader and baseline schema

Each database is created only through `load_week3_output`. The loader calls `create_sqlite_engine` and `create_all_tables`, then upserts devices, sessions, readings, and alerts. No benchmark index exists during the first measurement stage. SQLite's schema indexes for string primary keys and the sensor-reading uniqueness constraint remain in place; removing those would no longer test the real schema.

In [27]:
BENCHMARK_INDEX_DDL = {
    "idx_bench_reading_session_timestamp": (
        "CREATE INDEX idx_bench_reading_session_timestamp "
        "ON sensor_reading(session_id, timestamp_ms)"
    ),
    "idx_bench_reading_health": (
        "CREATE INDEX idx_bench_reading_health "
        "ON sensor_reading(composite_health_score)"
    ),
    "idx_bench_reading_gps_timestamp": (
        "CREATE INDEX idx_bench_reading_gps_timestamp "
        "ON sensor_reading(gps_dropout_long, timestamp_ms)"
    ),
}

def schema_indexes(connection: sqlite3.Connection) -> pd.DataFrame:
    rows = []
    for table in ("device", "session", "sensor_reading", "alert"):
        for seq, name, unique, origin, partial in connection.execute(
            f"PRAGMA index_list('{table}')"
        ).fetchall():
            columns = [
                item[2]
                for item in connection.execute(
                    f"PRAGMA index_info('{name}')"
                ).fetchall()
            ]
            rows.append(
                {
                    "table": table,
                    "index": name,
                    "columns": ", ".join(columns),
                    "unique": bool(unique),
                    "origin": origin,
                    "partial": bool(partial),
                }
            )
    return pd.DataFrame(rows)

def build_database(target_rows: int) -> tuple[Path, float, float, int]:
    # Reload so a long-running notebook kernel cannot retain the old
    # pre-LoadResult implementation of load_week3_output.
    import importlib as _importlib
    import ingen_pydev.db.loader as _loader_module
    current_loader = _importlib.reload(_loader_module)
    device_count = target_rows // PROFILE_ROWS
    # A unique path avoids Windows file-lock conflicts with databases that
    # may still be open in an older notebook cell or external DB viewer.
    run_token = time.time_ns()
    db_path = OUTPUT_DIR / (
        f"week4_real_schema_{target_rows}_{run_token}.db"
    )
    load_results = []
    for device_number in range(device_count):
        device_id = f"aido_rover_{device_number:04d}"
        result = current_loader.load_week3_output(
            parquet_path=PARQUET_PATH,
            summary_json_path=SUMMARY_PATH,
            db_path=db_path,
            device_id=device_id,
            device_name=f"Aido Rover {device_number:04d}",
            product_anchor="Aido Rover",
        )
        if not isinstance(result, current_loader.LoadResult):
            raise RuntimeError(
                "load_week3_output did not return LoadResult; rerun the "
                "notebook imports and database-generation cells"
            )
        load_results.append(result)
        if device_count >= 10 and (device_number + 1) % 10 == 0:
            print(f"  loaded {device_number + 1}/{device_count} devices")
    elapsed = sum(result.elapsed_seconds for result in load_results)
    readings = sum(result.readings_processed for result in load_results)
    alerts = sum(result.alerts_processed for result in load_results)
    throughput = readings / elapsed
    with sqlite3.connect(db_path) as connection:
        actual = connection.execute(
            "SELECT COUNT(*) FROM sensor_reading"
        ).fetchone()[0]
        benchmark_indexes = connection.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='index' AND name LIKE 'idx_bench_%'"
        ).fetchall()
    assert actual == target_rows
    assert benchmark_indexes == []
    return db_path, elapsed, throughput, alerts

## Five queries

Q1 and Q2 intentionally exercise indexes already guaranteed by the real schema. They are controls: supplemental indexes should not materially improve them. Q3–Q5 test plausible gaps. This mixture prevents the comparison from implying that every added index helps every workload.

In [28]:
def query_workload(
    connection: sqlite3.Connection,
) -> dict[str, tuple[str, tuple[Any, ...]]]:
    device_id = "aido_rover_0000"
    session_id = connection.execute(
        "SELECT session_id FROM session WHERE device_id = ?", (device_id,)
    ).fetchone()[0]
    start_ms, end_ms = connection.execute(
        "SELECT MIN(timestamp_ms), MAX(timestamp_ms) FROM sensor_reading "
        "WHERE device_id = ?",
        (device_id,),
    ).fetchone()
    reading_id = connection.execute(
        "SELECT reading_id FROM sensor_reading "
        "WHERE device_id = ? AND timestamp_ms = ?",
        (device_id, start_ms),
    ).fetchone()[0]
    range_start = end_ms - max(1, (end_ms - start_ms) // 10)
    return {
        "Q1 primary-key reading lookup": (
            "SELECT * FROM sensor_reading WHERE reading_id = ?",
            (reading_id,),
        ),
        "Q2 device/time unique-index range": (
            "SELECT timestamp_ms, battery_soc, composite_health_score "
            "FROM sensor_reading WHERE device_id = ? "
            "AND timestamp_ms BETWEEN ? AND ? ORDER BY timestamp_ms",
            (device_id, range_start, end_ms),
        ),
        "Q3 session/time range": (
            "SELECT timestamp_ms, battery_soc, lidar_distance_m "
            "FROM sensor_reading WHERE session_id = ? "
            "AND timestamp_ms BETWEEN ? AND ? ORDER BY timestamp_ms",
            (session_id, range_start, end_ms),
        ),
        "Q4 low-health aggregate": (
            "SELECT COUNT(*), MIN(composite_health_score), "
            "AVG(composite_health_score) FROM sensor_reading "
            "WHERE composite_health_score < ?",
            (60.0,),
        ),
        "Q5 GPS-dropout time range": (
            "SELECT COUNT(*), MIN(timestamp_ms), MAX(timestamp_ms) "
            "FROM sensor_reading WHERE gps_dropout_long = 1 "
            "AND timestamp_ms BETWEEN ? AND ?",
            (start_ms, end_ms),
        ),
    }

## Measurement helpers

Each measurement uses one untimed warm-up and the median of repeated executions. Result rows are retained for exact equality checks. `EXPLAIN QUERY PLAN` is recorded in full; the notebook does not reduce plans to a single selected line.

In [29]:
def explain_query_plan(
    connection: sqlite3.Connection,
    sql: str,
    parameters: tuple[Any, ...],
) -> str:
    plan_rows = connection.execute(
        "EXPLAIN QUERY PLAN " + sql, parameters
    ).fetchall()
    return " | ".join(str(row) for row in plan_rows)

def timed_query(
    connection: sqlite3.Connection,
    sql: str,
    parameters: tuple[Any, ...],
    repeats: int,
) -> tuple[float, list[tuple[Any, ...]]]:
    connection.execute(sql, parameters).fetchall()
    samples_ms = []
    result: list[tuple[Any, ...]] = []
    for _ in range(repeats):
        started_ns = time.perf_counter_ns()
        result = connection.execute(sql, parameters).fetchall()
        samples_ms.append((time.perf_counter_ns() - started_ns) / 1_000_000)
    return statistics.median(samples_ms), result

def measure_stage(
    connection: sqlite3.Connection, target_rows: int, stage: str
) -> tuple[list[dict[str, Any]], dict[str, list[tuple[Any, ...]]]]:
    repeats = 9 if target_rows < 1_000_000 else 5
    measurements = []
    results = {}
    for query_name, (sql, parameters) in query_workload(connection).items():
        latency_ms, result = timed_query(
            connection, sql, parameters, repeats
        )
        measurements.append(
            {
                "rows": target_rows,
                "query": query_name,
                "stage": stage,
                "median_latency_ms": latency_ms,
                "result_rows": len(result),
                "query_plan": explain_query_plan(
                    connection, sql, parameters
                ),
            }
        )
        results[query_name] = result
    return measurements, results

## Run the end-to-end benchmark

This cell can take a substantial amount of time because the one-million-row database is loaded through the production-style batched upsert path 100 times. That cost is part of the end-to-end setup and is recorded separately from query latency.

In [30]:
all_measurements: list[dict[str, Any]] = []
build_records: list[dict[str, Any]] = []
correctness_records: list[dict[str, Any]] = []
baseline_index_records: list[pd.DataFrame] = []
indexed_index_records: list[pd.DataFrame] = []

for target_rows in TARGET_ROWS:
    print(f"Building {target_rows:,} rows through load_week3_output ...")
    db_path, loader_seconds, loader_rows_per_second, alerts = (
        build_database(target_rows)
    )
    with sqlite3.connect(db_path) as connection:
        connection.execute("PRAGMA temp_store = MEMORY")
        baseline_indexes = schema_indexes(connection)
        baseline_indexes.insert(0, "rows", target_rows)
        baseline_index_records.append(baseline_indexes)

        before_metrics, before_results = measure_stage(
            connection, target_rows, "baseline schema"
        )
        index_started = time.perf_counter()
        for ddl in BENCHMARK_INDEX_DDL.values():
            connection.execute(ddl)
        connection.commit()
        index_seconds = time.perf_counter() - index_started

        indexed_indexes = schema_indexes(connection)
        indexed_indexes.insert(0, "rows", target_rows)
        indexed_index_records.append(indexed_indexes)
        after_metrics, after_results = measure_stage(
            connection, target_rows, "supplemental indexes"
        )

    all_measurements.extend(before_metrics + after_metrics)
    build_records.append(
        {
            "rows": target_rows,
            "database_path": str(db_path),
            "devices_loaded": target_rows // PROFILE_ROWS,
            "loader_seconds": loader_seconds,
            "loader_rows_per_second": loader_rows_per_second,
            "alerts_processed": alerts,
            "supplemental_index_build_seconds": index_seconds,
            "final_database_mb": db_path.stat().st_size / 1_000_000,
        }
    )
    for query_name, before_result in before_results.items():
        correctness_records.append(
            {
                "rows": target_rows,
                "query": query_name,
                "results_identical": before_result == after_results[query_name],
            }
        )
    del before_results, after_results
    gc.collect()
    print(
        f"Finished {target_rows:,}: loader={loader_seconds:.2f}s, "
        f"throughput={loader_rows_per_second:,.0f} rows/s, "
        f"supplemental indexes={index_seconds:.2f}s"
    )

measurements_df = pd.DataFrame(all_measurements)
builds_df = pd.DataFrame(build_records)
correctness_df = pd.DataFrame(correctness_records)
baseline_indexes_df = pd.concat(baseline_index_records, ignore_index=True)
indexed_indexes_df = pd.concat(indexed_index_records, ignore_index=True)
display(builds_df)

Building 10,000 rows through load_week3_output ...
Finished 10,000: loader=0.26s, throughput=37,810 rows/s, supplemental indexes=0.04s
Building 100,000 rows through load_week3_output ...
  loaded 10/10 devices
Finished 100,000: loader=4.69s, throughput=21,334 rows/s, supplemental indexes=0.29s
Building 1,000,000 rows through load_week3_output ...
  loaded 10/100 devices
  loaded 20/100 devices
  loaded 30/100 devices
  loaded 40/100 devices
  loaded 50/100 devices
  loaded 60/100 devices
  loaded 70/100 devices
  loaded 80/100 devices
  loaded 90/100 devices
  loaded 100/100 devices
Finished 1,000,000: loader=170.96s, throughput=5,849 rows/s, supplemental indexes=4.81s


,rows,database_path,devices_loaded,loader_seconds,loader_rows_per_second,alerts_processed,supplemental_index_build_seconds,final_database_mb
0,10000,outputs\w04_query_benchmarks\week4_real_schema...,1,0.264479,37810.128124,662,0.035257,5.787648
1,100000,outputs\w04_query_benchmarks\week4_real_schema...,10,4.687369,21333.931821,6620,0.291926,57.933824
2,1000000,outputs\w04_query_benchmarks\week4_real_schema...,100,170.959966,5849.322635,66200,4.813463,579.465216


## Baseline-index evidence

The baseline table below is evidence that the real primary-key and unique-constraint indexes were retained. `origin = pk` identifies a primary-key index and `origin = u` identifies a unique-constraint index. The second table includes those same indexes plus the three `idx_bench_...` additions.

In [31]:
display(baseline_indexes_df)
display(indexed_indexes_df)
assert not baseline_indexes_df["index"].str.startswith("idx_bench_").any()
assert indexed_indexes_df["index"].str.startswith("idx_bench_").any()

,rows,table,index,columns,unique,origin,partial
0,10000,device,sqlite_autoindex_device_1,device_id,True,pk,False
1,10000,session,sqlite_autoindex_session_1,session_id,True,pk,False
2,10000,sensor_reading,sqlite_autoindex_sensor_reading_2,"device_id, timestamp_ms",True,u,False
3,10000,sensor_reading,sqlite_autoindex_sensor_reading_1,reading_id,True,pk,False
4,10000,alert,sqlite_autoindex_alert_1,alert_id,True,pk,False
5,100000,device,sqlite_autoindex_device_1,device_id,True,pk,False
6,100000,session,sqlite_autoindex_session_1,session_id,True,pk,False
7,100000,sensor_reading,sqlite_autoindex_sensor_reading_2,"device_id, timestamp_ms",True,u,False
8,100000,sensor_reading,sqlite_autoindex_sensor_reading_1,reading_id,True,pk,False
9,100000,alert,sqlite_autoindex_alert_1,alert_id,True,pk,False


,rows,table,index,columns,unique,origin,partial
0,10000,device,sqlite_autoindex_device_1,device_id,True,pk,False
1,10000,session,sqlite_autoindex_session_1,session_id,True,pk,False
2,10000,sensor_reading,idx_bench_reading_gps_timestamp,"gps_dropout_long, timestamp_ms",False,c,False
3,10000,sensor_reading,idx_bench_reading_health,composite_health_score,False,c,False
4,10000,sensor_reading,idx_bench_reading_session_timestamp,"session_id, timestamp_ms",False,c,False
5,10000,sensor_reading,sqlite_autoindex_sensor_reading_2,"device_id, timestamp_ms",True,u,False
6,10000,sensor_reading,sqlite_autoindex_sensor_reading_1,reading_id,True,pk,False
7,10000,alert,sqlite_autoindex_alert_1,alert_id,True,pk,False
8,100000,device,sqlite_autoindex_device_1,device_id,True,pk,False
9,100000,session,sqlite_autoindex_session_1,session_id,True,pk,False


## Correctness

Indexes must not change results. All 15 scale/query comparisons are shown, not summarized selectively.

In [32]:
display(correctness_df)
assert correctness_df["results_identical"].all()
print("Confirmed: all 15 before/after results are exactly identical.")

,rows,query,results_identical
0,10000,Q1 primary-key reading lookup,True
1,10000,Q2 device/time unique-index range,True
2,10000,Q3 session/time range,True
3,10000,Q4 low-health aggregate,True
4,10000,Q5 GPS-dropout time range,True
5,100000,Q1 primary-key reading lookup,True
6,100000,Q2 device/time unique-index range,True
7,100000,Q3 session/time range,True
8,100000,Q4 low-health aggregate,True
9,100000,Q5 GPS-dropout time range,True


Confirmed: all 15 before/after results are exactly identical.


## Q2 — before/after assessment

This comparison reports **every query at every scale**, including slowdowns and changes within ±5% that are reasonably classified as measurement noise/no material change. It does not select only the largest speedup. Q1 and Q2 are expected to show little improvement because the baseline schema already indexes their access paths. A supplemental index can also make no difference, or occasionally benchmark slower, due to cache effects and planner choices.

In [33]:
latency_comparison = (
    measurements_df.pivot(
        index=["rows", "query"],
        columns="stage",
        values="median_latency_ms",
    )
    .reset_index()
    .rename_axis(columns=None)
)
latency_comparison["speedup_x"] = (
    latency_comparison["baseline schema"]
    / latency_comparison["supplemental indexes"]
)
latency_comparison["latency_change_pct"] = (
    latency_comparison["supplemental indexes"]
    / latency_comparison["baseline schema"]
    - 1
) * 100

def honest_assessment(speedup: float) -> str:
    if speedup > 1.05:
        return "faster with supplemental indexes"
    if speedup < 0.95:
        return "slower with supplemental indexes"
    return "no material change (within ±5%)"

latency_comparison["assessment"] = latency_comparison["speedup_x"].map(
    honest_assessment
)
display(
    latency_comparison.style.format(
        {
            "baseline schema": "{:.4f}",
            "supplemental indexes": "{:.4f}",
            "speedup_x": "{:.2f}x",
            "latency_change_pct": "{:+.1f}%",
        }
    )
)
display(
    latency_comparison.groupby(["query", "assessment"], dropna=False)
    .size()
    .rename("scale_count")
    .reset_index()
)

,rows,query,baseline schema,supplemental indexes,speedup_x,latency_change_pct,assessment
0,10000,Q1 primary-key reading lookup,0.0244,0.0286,0.85x,+17.2%,slower with supplemental indexes
1,10000,Q2 device/time unique-index range,0.5931,0.5962,0.99x,+0.5%,no material change (within ±5%)
2,10000,Q3 session/time range,2.6328,0.5342,4.93x,-79.7%,faster with supplemental indexes
3,10000,Q4 low-health aggregate,1.9672,0.0224,87.82x,-98.9%,faster with supplemental indexes
4,10000,Q5 GPS-dropout time range,1.7962,0.0222,80.91x,-98.8%,faster with supplemental indexes
5,100000,Q1 primary-key reading lookup,0.0320,0.0285,1.12x,-10.9%,faster with supplemental indexes
6,100000,Q2 device/time unique-index range,0.7816,0.6936,1.13x,-11.3%,faster with supplemental indexes
7,100000,Q3 session/time range,40.9886,0.6149,66.66x,-98.5%,faster with supplemental indexes
8,100000,Q4 low-health aggregate,37.3719,0.0463,807.17x,-99.9%,faster with supplemental indexes
9,100000,Q5 GPS-dropout time range,34.0001,0.0213,1596.25x,-99.9%,faster with supplemental indexes


,query,assessment,scale_count
0,Q1 primary-key reading lookup,faster with supplemental indexes,1
1,Q1 primary-key reading lookup,no material change (within ±5%),1
2,Q1 primary-key reading lookup,slower with supplemental indexes,1
3,Q2 device/time unique-index range,faster with supplemental indexes,2
4,Q2 device/time unique-index range,no material change (within ±5%),1
5,Q3 session/time range,faster with supplemental indexes,3
6,Q4 low-health aggregate,faster with supplemental indexes,3
7,Q5 GPS-dropout time range,faster with supplemental indexes,3


## Full query-plan comparison

Plans are shown side by side for all queries and scales. The `plan_changed` flag describes planner output only; it is not treated as proof of a speedup.

In [34]:
plan_comparison = (
    measurements_df.pivot(
        index=["rows", "query"], columns="stage", values="query_plan"
    )
    .reset_index()
    .rename_axis(columns=None)
)
plan_comparison["plan_changed"] = (
    plan_comparison["baseline schema"]
    != plan_comparison["supplemental indexes"]
)
display(plan_comparison)

,rows,query,baseline schema,supplemental indexes,plan_changed
0,10000,Q1 primary-key reading lookup,"(3, 0, 39, 'SEARCH sensor_reading USING INDEX ...","(3, 0, 39, 'SEARCH sensor_reading USING INDEX ...",False
1,10000,Q2 device/time unique-index range,"(4, 0, 49, 'SEARCH sensor_reading USING INDEX ...","(4, 0, 49, 'SEARCH sensor_reading USING INDEX ...",False
2,10000,Q3 session/time range,"(3, 0, 216, 'SCAN sensor_reading') | (18, 0, 0...","(4, 0, 49, 'SEARCH sensor_reading USING INDEX ...",True
3,10000,Q4 low-health aggregate,"(3, 0, 216, 'SCAN sensor_reading')","(3, 0, 186, 'SEARCH sensor_reading USING COVER...",True
4,10000,Q5 GPS-dropout time range,"(3, 0, 216, 'SCAN sensor_reading')","(3, 0, 45, 'SEARCH sensor_reading USING COVERI...",True
5,100000,Q1 primary-key reading lookup,"(3, 0, 39, 'SEARCH sensor_reading USING INDEX ...","(3, 0, 39, 'SEARCH sensor_reading USING INDEX ...",False
6,100000,Q2 device/time unique-index range,"(4, 0, 49, 'SEARCH sensor_reading USING INDEX ...","(4, 0, 49, 'SEARCH sensor_reading USING INDEX ...",False
7,100000,Q3 session/time range,"(3, 0, 216, 'SCAN sensor_reading') | (18, 0, 0...","(4, 0, 49, 'SEARCH sensor_reading USING INDEX ...",True
8,100000,Q4 low-health aggregate,"(3, 0, 216, 'SCAN sensor_reading')","(3, 0, 186, 'SEARCH sensor_reading USING COVER...",True
9,100000,Q5 GPS-dropout time range,"(3, 0, 216, 'SCAN sensor_reading')","(3, 0, 45, 'SEARCH sensor_reading USING COVERI...",True


## Save all evidence

The exports retain raw timings, every before/after comparison, full plans, correctness results, loader/index-build costs, and both index inventories.

In [35]:
measurements_df.to_csv(OUTPUT_DIR / "raw_measurements.csv", index=False)
latency_comparison.to_csv(OUTPUT_DIR / "latency_comparison.csv", index=False)
plan_comparison.to_csv(OUTPUT_DIR / "query_plan_comparison.csv", index=False)
correctness_df.to_csv(OUTPUT_DIR / "correctness_checks.csv", index=False)
builds_df.to_csv(OUTPUT_DIR / "database_builds.csv", index=False)
baseline_indexes_df.to_csv(OUTPUT_DIR / "baseline_indexes.csv", index=False)
indexed_indexes_df.to_csv(OUTPUT_DIR / "indexed_indexes.csv", index=False)
print(f"Saved complete benchmark evidence to {OUTPUT_DIR.resolve()}")

Saved complete benchmark evidence to C:\ingen\outputs\w04_query_benchmarks
